# TTS Text Optimizer with Thinking Models

**Format translated text for optimal DesiVocal.com TTS output using reasoning-capable LLMs.**

This notebook:
1. **Installs Ollama** and downloads a thinking model (deepseek-r1, qwen3, magistral, etc.)
2. **Shows the model's reasoning process** in real-time as it analyzes your text
3. **Formats text** with proper speaker identification, punctuation, and DesiVocal-specific fixes
4. **Downloads** the optimized `.txt` file ready for TTS

Optimized for translating public-domain literary works (Sherlock Holmes, Ramayana, Mahabharata, Dracula, Alice in Wonderland, Pride and Prejudice, etc.) into natural, human-like TTS audio.

## Step 1: Install and Setup Ollama
Run this cell to install Ollama and start the server in the background.

In [ ]:
# Install required packages
!pip install -q ollama requests ipywidgets

# Install and start Ollama server
import subprocess
import time
import os
import sys

print("Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\nStarting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("[OK] Ollama server is running and ready!")
except Exception as e:
    print(f"[WARN] Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")

## Step 2: Select and Download Thinking Model
Choose a reasoning-capable model. These models show their thinking process before producing output.

**Recommended:**
- `deepseek-r1:14b` - Best reasoning quality for the size
- `qwen3:14b` - Strong multilingual reasoning
- `magistral:24b` - Mistral's reasoning model (needs more VRAM)
- `gpt-oss:20b` - OpenAI-style thinking format

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import ollama

print("Thinking Model Selection")
print("=" * 40)

# Thinking model options
THINKING_MODELS = {
    "deepseek-r1:14b (Best Reasoning)": "deepseek-r1:14b",
}

model_dropdown = widgets.Dropdown(
    options=list(THINKING_MODELS.keys()),
    value="deepseek-r1:14b (Best Reasoning)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

display(model_dropdown)
print("\nSelect a model and run the next cell to download it.")
print("NOTE: 14b models need ~10GB VRAM, 24b models need ~16GB VRAM.")

In [ ]:
# Pull the selected thinking model
selected_model_name = THINKING_MODELS[model_dropdown.value]
print(f"Downloading model: {selected_model_name}...")
print("This may take several minutes for large models.")

try:
    current_digest = ''
    for progress in ollama.pull(selected_model_name, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
             print()
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
             completed = progress['completed']
             total = progress['total']
             pct = (completed / total * 100) if total > 0 else 0
             print(f"\r   {status}: {pct:.1f}%", end='', flush=True)
        else:
             print(f"\r   {status}", end='', flush=True)

    print(f"\n\n[OK] Model '{selected_model_name}' ready to use!")
except Exception as e:
    print(f"\n[ERROR] Error pulling model: {e}")

## Step 3: TTS Optimizer Engine (with Thinking Display)
This defines the optimizer class that streams the model's thinking process in real-time.

In [ ]:
import requests
import json
import sys
import re
import time
from IPython.display import display, HTML, clear_output

# ══════════════════════════════════════════════════════════════
# KNOWN THINKING MODEL PATTERNS
# ══════════════════════════════════════════════════════════════
# Different thinking models use different tag patterns for their reasoning:
#   deepseek-r1  : <think> ... </think>
#   qwen3        : <think> ... </think>
#   magistral    : [Thinking] ... [/Thinking] or just outputs reasoning first
#   gpt-oss      : <|begin_of_thought|> ... <|end_of_thought|>
# We handle all of these patterns.

THINK_START_PATTERNS = ['<think>', '<|begin_of_thought|>', '[Thinking]']
THINK_END_PATTERNS = ['</think>', '<|end_of_thought|>', '[/Thinking]']


class TTSThinkingOptimizer:
    """Optimizes text for DesiVocal.com TTS using thinking models with visible reasoning."""

    def __init__(self, model_name="deepseek-r1:14b", chunk_size=2000, timeout=6000):
        self.ollama_url = "http://localhost:11434/api/generate"
        self.model = model_name
        self.chunk_size = chunk_size
        self.timeout = timeout
        print(f"Initialized TTS Thinking Optimizer")
        print(f"   Model: {self.model}")
        print(f"   Chunk size: {self.chunk_size} chars")
        print(f"   Timeout: {self.timeout}s per chunk")

    def chunk_text(self, text: str) -> list:
        """Split text into chunks at paragraph/sentence boundaries."""
        if len(text) <= self.chunk_size:
            return [text]

        chunks = []
        current_chunk = ""

        # First try to split by paragraphs (double newline)
        paragraphs = text.split('\n\n')

        for para in paragraphs:
            if len(current_chunk) + len(para) + 2 > self.chunk_size and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = para
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        # If any chunk is still too large, split by sentences
        final_chunks = []
        for chunk in chunks:
            if len(chunk) <= self.chunk_size:
                final_chunks.append(chunk)
            else:
                sentences = re.split(r'([।॥.!?]\s+)', chunk)
                sub_chunk = ""
                for i in range(0, len(sentences), 2):
                    sentence = sentences[i]
                    separator = sentences[i+1] if i+1 < len(sentences) else ""
                    if len(sub_chunk) + len(sentence) + len(separator) > self.chunk_size and sub_chunk:
                        final_chunks.append(sub_chunk.strip())
                        sub_chunk = sentence + separator
                    else:
                        sub_chunk += sentence + separator
                if sub_chunk.strip():
                    final_chunks.append(sub_chunk.strip())

        # Fallback: if still no chunks, force-split
        if not final_chunks:
            final_chunks = [text[i:i+self.chunk_size] for i in range(0, len(text), self.chunk_size)]

        print(f"\nText split into {len(final_chunks)} chunks")
        for idx, chunk in enumerate(final_chunks, 1):
            print(f"   Chunk {idx}: {len(chunk)} characters")

        return final_chunks

    def get_optimization_prompt(self, text: str) -> str:
        """Build the TTS optimization prompt for thinking models."""
        prompt = f"""
        You are an expert text formatter for DesiVocal.com TTS system with advanced reasoning capabilities.

═══════════════════════════════════════════════════════════════
CRITICAL INSTRUCTIONS FOR REASONING MODELS
═══════════════════════════════════════════════════════════════

You are a reasoning model. Use your thinking process INTERNALLY to:
- Identify characters and their relationships
- Resolve pronouns and track conversations
- Assign appropriate punctuation to each speaker
- Apply technical formatting correctly

HOWEVER: Your reasoning process MUST NOT appear in the final output.

FORBIDDEN IN OUTPUT:
- Your thinking process or reasoning steps
- Chinese text or any language other than Hindi
- Explanations like "I will now format..." or "Let me think..."
- Meta-commentary about the task
- Repetition of the same sentence
- English text (except technical terms that are already in English)

REQUIRED IN OUTPUT:
- ONLY the formatted Hindi text
- Nothing else

═══════════════════════════════════════════════════════════════
YOUR MISSION
═══════════════════════════════════════════════════════════════

Format Hindi text for DesiVocal.com - a single-voice TTS system without SSML support.

THE CHALLENGE:
One voice reads everything. Listeners cannot distinguish speakers unless explicitly marked.

YOUR SOLUTION:
Use DIFFERENT punctuation marks for DIFFERENT characters so listeners know who's speaking.

═══════════════════════════════════════════════════════════════
CORE RULE: WORD PRESERVATION
═══════════════════════════════════════════════════════════════

Every input word MUST appear in output.
Exception: Attribution words ("ne kaha", "ne poocha") become speaker tags.

ALLOWED:
- Format numbers, dates, punctuation
- ADD speaker tags
- ADD different dialogue punctuation per character

ABSOLUTELY FORBIDDEN:
- Adding explanations, annotations, metadata
- Translating between languages
- Dropping any words
- Repeating sentences
- Writing in Chinese
- Adding your own content
- Including your reasoning process

═══════════════════════════════════════════════════════════════
REASONING PROCESS (INTERNAL - DO NOT OUTPUT)
═══════════════════════════════════════════════════════════════

Use your reasoning capabilities to:

STEP 1: CHARACTER DISCOVERY
Scan text and identify:
- Named characters: "Holmes ne kaha" → Holmes is a character
- Roles/titles: raja, doctor, maharaj
- Pronouns: Track who "usne", "maine" refer to

STEP 2: PRONOUN RESOLUTION
When you see "usne kaha":
- Look back 1-3 sentences
- Identify the most recent/contextually relevant person
- Track conversation flow

STEP 3: CONVERSATION TRACKING
In dialogue:
- Speakers typically alternate
- Unless context indicates otherwise (interruption, monologue)
- Maintain consistency

STEP 4: PUNCTUATION ASSIGNMENT
Assign DIFFERENT marks to EACH character:
- Main protagonist → 'single quotes'
- Secondary/narrator → "double quotes"
- Authority/client → *asterisks*
- Others → <<guillemets>>

Stay consistent throughout entire text.

STEP 5: APPLY FORMATTING
- Convert Roman numerals
- Format years, dates, times
- Fix the "10" bug
- Add pacing punctuation

STEP 6: VERIFY
- No repetitions?
- All Hindi?
- No reasoning shown?
- Word count preserved?

REMEMBER: This reasoning is INTERNAL. Do not include any of these steps in your output.

═══════════════════════════════════════════════════════════════
SPEAKER IDENTIFICATION
═══════════════════════════════════════════════════════════════

Look for context clues:
"Holmes ne kaha" → Speaker is Holmes
"raj ne poocha" → Speaker is raj
"maine kaha" → Speaker is narrator (main)
"usne kaha" → Use reasoning to identify who

Priority:
1. Named characters (Holmes, Watson, Ram, Sita)
2. Roles/titles (raja, doctor, maharaj)
3. Context-based pronoun resolution
4. Last resort: vakta1, vakta2 (avoid if possible)

Track conversations: Speakers alternate in dialogue.

For Sherlock Holmes stories:
- Holmes = protagonist ('single quotes')
- Watson = narrator/sidekick ("double quotes")
- Clients/visitors = (*asterisks*)
- Others = (<<guillemets>>)

═══════════════════════════════════════════════════════════════
DIALOGUE FORMATTING
═══════════════════════════════════════════════════════════════

FORMAT:
SpeakerName: [mark]dialogue[mark]

EXAMPLE:
Holmes: 'yah zaroori hai.'
Watson: "samajh gaya."
raja: *madad karo.*
servant: <<ji maharaj.>>

CONSISTENCY: Each character keeps same punctuation throughout.

REMOVE ATTRIBUTION: "Holmes ne kaha" → "Holmes:"

SEPARATION: Each speaker on new line with blank line before.

═══════════════════════════════════════════════════════════════
TECHNICAL FORMATTING RULES
═══════════════════════════════════════════════════════════════

1. ROMAN NUMERALS → NUMBERS
I→1, II→2, III→3, IV→4, V→5, VI→6, VII→7, VIII→8, IX→9, X→10
Chapter I → Chapter 1
adhyay II → adhyay 2

2. NUMBERS: Remove commas
50,000 → 50000
1,50,000 → 150000

3. DATES: Month names
15/03/2024 → 15 March 2024 or 15 मार्च 2024

4. YEARS: "san" prefix (no space)
1988 → san1988
"main 1995 mein paida hua" → "main san1995 mein paida hua"

5. TIME: Write in words
3:30 → saadhe teen
10:00 → ten baje

6. THE "10" BUG - CRITICAL
DesiVocal doesn't speak "10" or "दस" properly!
ALWAYS use "ten":
10 books → ten kitaabein
Chapter 10 → Chapter ten
10:00 → ten baje
8-10 → 8seten

7. RANGES: "se" (no space)
5-8 → 5se8
10-15 → tense15

8. PERCENTAGES
50% → 50 percent

9. ABBREVIATIONS
Dr. → Doctor or Daktar
Rs. → rupaye
km → kilometer

10. ACRONYMS: Remove periods
U.S.A. → USA
N.A.S.A. → NASA

11. EMAILS/URLS
@ → at the rate
. → dot (in email/URL only)
hr@company.com → hr at the rate company dot com

12. HYPHENS: Remove from compounds
cross-check → cross check

13. SYMBOLS
°F → degree Fahrenheit
× → guna

═══════════════════════════════════════════════════════════════
PACING PUNCTUATION (Non-dialogue)
═══════════════════════════════════════════════════════════════

, = short pause
| = medium pause (context shift)
. = long pause (sentence end)
,, = extended pause
... = suspense
!! = excitement
?? = confusion

Example: "usne khana khaya. | fir so gaya. | subah utha."

═══════════════════════════════════════════════════════════════
EXAMPLES (STUDY CAREFULLY)
═══════════════════════════════════════════════════════════════

EXAMPLE 1: Sherlock Holmes

INPUT:
adhyay I
yah 15 march, 1988 ki baat hai. Holmes ne kaha, "main 10 baje aaunga." Watson ne poocha, "kyun?" "kyunki yah zaroori hai," Holmes ne kaha.

OUTPUT:
adhyay 1.

yah 15 march, san1988 ki baat hai.

Holmes: 'main ten baje aaunga.'

Watson: "kyun?"

Holmes: 'kyunki yah zaroori hai.'

---

EXAMPLE 2: Pronoun resolution

INPUT:
Holmes kamre mein khada tha. Watson darwaze par aaya. usne poocha, "kya hua?" "kuch nahi," usne kaha.

OUTPUT:
Holmes kamre mein khada tha. | Watson darwaze par aaya.

Watson: "kya hua?"

Holmes: 'kuch nahi.'

---

EXAMPLE 3: Multiple characters

INPUT:
raja ne kaha, "madad karo." Holmes ne kaha, "main karunga." Watson ne kaha, "main bhi saath chalunga."

OUTPUT:
raja: *madad karo.*

Holmes: 'main karunga.'

Watson: "main bhi saath chalunga."

═══════════════════════════════════════════════════════════════
VERIFICATION CHECKLIST (INTERNAL)
═══════════════════════════════════════════════════════════════

Before outputting, verify:
✓ All Roman numerals converted (I→1, II→2)
✓ All "10" → "ten"
✓ All years have "san" prefix
✓ Numbers without commas
✓ Speakers identified from context
✓ Each character has consistent punctuation
✓ Attribution words removed
✓ Email/URL dots → "dot"
✓ NO REPETITION
✓ NO CHINESE TEXT
✓ NO REASONING SHOWN
✓ ALL HINDI

═══════════════════════════════════════════════════════════════
OUTPUT FORMAT
═══════════════════════════════════════════════════════════════

CRITICAL FOR REASONING MODELS:

Your output must contain ONLY the formatted Hindi text.

DO NOT include:
- Your thinking process
- "Let me analyze..."
- "First, I will..."
- "The characters are..."
- Any Chinese characters (不要用中文)
- Any English explanations
- "Here is the formatted text:"
- Reasoning steps
- Meta-commentary

DO include:
- The formatted Hindi text
- Nothing else

═══════════════════════════════════════════════════════════════
ANTI-HALLUCINATION GUARDS
═══════════════════════════════════════════════════════════════

1. NEVER repeat the same sentence more than once
2. NEVER write in Chinese (不要用中文)
3. NEVER write in English (except existing English words in input)
4. NEVER add your reasoning to the output
5. NEVER add explanations or meta-commentary
6. If you notice yourself repeating, STOP immediately

═══════════════════════════════════════════════════════════════
INPUT TEXT:
{text}"""
        return prompt

    def optimize_chunk_streaming(self, chunk: str, chunk_num: int = 1, total_chunks: int = 1, retry_count: int = 3) -> str:
        """
        Optimize a single chunk with streaming output showing thinking process.
        """
        prompt = self.get_optimization_prompt(chunk)

        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": True,
            "options": {
                "temperature": 0.3,
                "top_p": 0.9,
                "num_predict": -1
            }
        }

        for attempt in range(retry_count):
            try:
                response = requests.post(
                    self.ollama_url,
                    json=payload,
                    stream=True,
                    timeout=self.timeout
                )
                response.raise_for_status()

                full_response = ""
                thinking_content = ""
                output_content = ""
                in_thinking = False
                thinking_started = False
                thinking_ended = False

                print(f"\n{'='*60}")
                print(f"  CHUNK {chunk_num}/{total_chunks} ({len(chunk)} chars)")
                print(f"{'='*60}")

                for line in response.iter_lines():
                    if not line:
                        continue
                    try:
                        data = json.loads(line)
                        token = data.get("response", "")
                        full_response += token

                        # Detect thinking start
                        for pattern in THINK_START_PATTERNS:
                            if pattern in full_response and not thinking_started:
                                thinking_started = True
                                in_thinking = True
                                print(f"\n--- MODEL REASONING ---")
                                # Remove the tag from display
                                remaining = full_response.split(pattern, 1)[-1]
                                if remaining:
                                    thinking_content += remaining
                                    sys.stdout.write(remaining)
                                    sys.stdout.flush()
                                full_response = ""  # Reset to avoid re-matching
                                break

                        # Detect thinking end
                        if in_thinking:
                            for pattern in THINK_END_PATTERNS:
                                if pattern in token:
                                    in_thinking = False
                                    thinking_ended = True
                                    # Get any content before the end tag
                                    before_tag = token.split(pattern)[0]
                                    if before_tag:
                                        thinking_content += before_tag
                                        sys.stdout.write(before_tag)
                                        sys.stdout.flush()
                                    print(f"\n--- END REASONING ---\n")
                                    print(f"--- FORMATTED OUTPUT ---")
                                    # Get content after end tag
                                    after_tag = token.split(pattern, 1)[-1]
                                    if after_tag.strip():
                                        output_content += after_tag
                                        sys.stdout.write(after_tag)
                                        sys.stdout.flush()
                                    break
                            else:
                                if in_thinking:
                                    thinking_content += token
                                    sys.stdout.write(token)
                                    sys.stdout.flush()
                        elif thinking_ended:
                            # We are past thinking, collecting output
                            output_content += token
                            sys.stdout.write(token)
                            sys.stdout.flush()
                        elif not thinking_started:
                            # Model might not use thinking tags, treat as direct output
                            output_content += token
                            sys.stdout.write(token)
                            sys.stdout.flush()

                        if data.get("done", False):
                            break
                    except json.JSONDecodeError:
                        continue

                print(f"\n{'='*60}")

                # If we never entered thinking mode, the full response is the output
                if not thinking_started:
                    output_content = full_response

                # Clean the output
                cleaned = self._clean_output(output_content)
                print(f"[OK] Chunk {chunk_num}/{total_chunks} complete! ({len(cleaned)} chars output)")

                return cleaned

            except requests.exceptions.Timeout:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 10
                    print(f"\n[WARN] Timeout on attempt {attempt + 1}/{retry_count}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n[ERROR] Failed after {retry_count} attempts due to timeout")
                    raise
            except Exception as e:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 5
                    print(f"\n[WARN] Error on attempt {attempt + 1}/{retry_count}: {e}")
                    print(f"   Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n[ERROR] Failed after {retry_count} attempts: {e}")
                    raise

        return None

    def optimize(self, text: str) -> str:
        """Optimize text with automatic chunking, streaming thinking display."""
        chunks = self.chunk_text(text)

        if len(chunks) == 1:
            print(f"\nProcessing single chunk ({len(text)} chars)...")
            return self.optimize_chunk_streaming(chunks[0], 1, 1)

        print(f"\nProcessing {len(chunks)} chunks with visible reasoning...")
        optimized_chunks = []

        for idx, chunk in enumerate(chunks, 1):
            try:
                optimized = self.optimize_chunk_streaming(chunk, idx, len(chunks))
                if optimized:
                    optimized_chunks.append(optimized)
                else:
                    print(f"[WARN] Chunk {idx}/{len(chunks)} failed - using original")
                    optimized_chunks.append(chunk)
            except Exception as e:
                print(f"[ERROR] Error processing chunk {idx}: {e}")
                print("   Using original chunk text")
                optimized_chunks.append(chunk)

        final_text = "\n\n".join(optimized_chunks)
        print(f"\n[OK] All chunks processed! Total output: {len(final_text)} characters")
        return final_text

    def _clean_output(self, text: str) -> str:
        """Clean the model output to remove formatting artifacts."""
        # Remove markdown artifacts
        text = text.replace("```", "").replace("**", "")

        # Remove any remaining thinking tags
        for pattern in THINK_START_PATTERNS + THINK_END_PATTERNS:
            text = text.replace(pattern, "")

        # Remove lines that look like metadata/headers
        lines = []
        for line in text.split('\n'):
            stripped = line.strip()
            if stripped and not stripped.startswith('#') and not stripped.startswith('OUTPUT'):
                # Skip lines that are just dashes or equals
                if not re.match(r'^[-=]{3,}$', stripped):
                    lines.append(line.rstrip())

        return '\n'.join(lines).strip()


print("[OK] TTSThinkingOptimizer class loaded with streaming reasoning support!")

## Step 4: Upload Text and Configure
Upload your `.txt` file and set the chunk size. The model's thinking process will be displayed as it works.

In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

print("Upload your text file (.txt):")
uploaded = files.upload()

if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    file_size = len(uploaded[uploaded_filename])
    print(f"[OK] Uploaded: {uploaded_filename} ({file_size:,} bytes)")
else:
    print("[WARN] No file uploaded yet.")

# Chunk size selector
chunk_size_input = widgets.IntSlider(
    value=2000,
    min=500,
    max=5000,
    step=100,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

print("\nConfiguration:")
display(chunk_size_input)
print("\nChunk size guide: smaller = more API calls but less timeout risk")
print("   Recommended: 1500-2500 for 14b models, 1000-1500 for 7b models")

## Step 5: Run Optimization (with Visible Thinking)
The model will show its reasoning process as it analyzes and formats each chunk.
You will see:
- `--- MODEL REASONING ---` : The model thinking through speaker identification, context analysis
- `--- FORMATTED OUTPUT ---` : The actual formatted text for TTS

In [ ]:
# Run Optimization with Thinking Display
if not uploaded:
    print("[WARN] Please upload a file in the previous step first!")
else:
    try:
        text_content = uploaded[uploaded_filename].decode("utf-8")
        print(f"Read {len(text_content):,} characters from file.")

        # Initialize optimizer with selected model
        try:
            model_to_use = selected_model_name
        except NameError:
            model_to_use = "deepseek-r1:14b"  # Fallback
            print("[WARN] Using default model: deepseek-r1:14b")

        optimizer = TTSThinkingOptimizer(
            model_name=model_to_use,
            chunk_size=chunk_size_input.value,
            timeout=6000  # 100 minutes per chunk for thinking models
        )

        print(f"\nStarting optimization with {model_to_use}...")
        print("The model's reasoning process will be displayed below.")
        print("=" * 60)

        start_time = time.time()
        optimized_text = optimizer.optimize(text_content)
        end_time = time.time()

        processing_time = end_time - start_time

        if optimized_text:
            print("\n" + "=" * 60)
            print("OPTIMIZATION COMPLETE!")
            print("=" * 60)
            print(f"Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")
            print(f"Input length: {len(text_content):,} chars")
            print(f"Output length: {len(optimized_text):,} chars")
            print(f"Size change: {((len(optimized_text) - len(text_content)) / len(text_content) * 100):+.1f}%")

            print("\nPreview (First 800 characters):")
            print("=" * 60)
            print(optimized_text[:800])
            if len(optimized_text) > 800:
                print("\n... (truncated)")
            print("=" * 60)

            # Save to file
            output_filename = f"tts_optimized_{uploaded_filename}"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(optimized_text)

            print(f"\nSaved to: {output_filename}")
            print("Downloading file...")

            # Trigger download
            files.download(output_filename)
            print("\n[OK] Done! Check your downloads folder.")

        else:
            print("\n[ERROR] Optimization failed. Please check the errors above.")

    except Exception as e:
        print(f"\n[ERROR] Error reading or processing file: {e}")
        import traceback
        print("\nFull error details:")
        print(traceback.format_exc())

## Troubleshooting

**If you get timeouts:**
1. Reduce chunk size to 1000-1500 characters
2. Use a smaller model: `deepseek-r1:7b` or `qwen3:8b`
3. Check Ollama server: `!ollama ps`
4. Restart Ollama: Go back to Step 1 and re-run

**Thinking models are slower** than regular models because they reason through the text first. This is expected and produces better results, especially for:
- Complex dialogue with many speakers
- Pronoun resolution in long passages
- Genre-appropriate formatting

**For very large files (100k+ chars):**
- Use chunk size of 1000 characters
- Consider splitting the file manually
- Processing will take longer but will be more reliable